In [1]:
!test -f obs_3day.bufr \
    || wget https://sites.ecmwf.int/repository/pdbufr/test-data/obs_3day.bufr \
             --output-document=obs_3day.bufr

# Flat reader: overview

In [2]:
import pdbufr

## Block extraction mode

Block extraction is activated by passing one of the special values `"all"`, `"header"` or `"data"` to the `columns` parameter.  The default (`columns=[]`) is equivalent to `columns="all"`.

### All keys

Extracting all keys returns every header and data key.  Column names carry the ecCodes rank prefix, e.g. `#1#latitude`.

In [3]:
df = pdbufr.read_bufr("obs_3day.bufr", columns="all", reader="flat")
print(f"{len(df)} rows, {len(df.columns)} columns")
df[["ident", "#1#latitude", "#1#longitude", "#1#airTemperatureAt2M"]].head()

50 rows, 103 columns


,ident,#1#latitude,#1#longitude,#1#airTemperatureAt2M
0,03894,49.43,-2.60,282.4
1,03590,52.12,0.96,281.7
2,03379,53.03,-0.50,280.7
3,03391,53.09,-0.17,281.0
4,03743,51.20,-1.81,281.2


### Header keys only

Passing `columns="header"` limits the output to header-section keys.  This is fast because the data section is not decoded.

In [4]:
df_hdr = pdbufr.read_bufr("obs_3day.bufr", columns="header", reader="flat")
print(f"{len(df_hdr)} rows, {len(df_hdr.columns)} columns")
df_hdr[["ident", "typicalDate", "typicalTime", "bufrHeaderCentre"]].head()

50 rows, 50 columns


,ident,typicalDate,typicalTime,bufrHeaderCentre
0,03894,20170425,120000,98
1,03590,20170425,120000,98
2,03379,20170425,120000,98
3,03391,20170425,120000,98
4,03743,20170425,120000,98


### Data keys only

Passing `columns="data"` returns only the data-section keys.

In [5]:
df_data = pdbufr.read_bufr("obs_3day.bufr", columns="data", reader="flat")
print(f"{len(df_data)} rows, {len(df_data.columns)} columns")
df_data[["#1#latitude", "#1#longitude", "#1#airTemperatureAt2M", "#1#windSpeedAt10M"]].head()

50 rows, 53 columns


,#1#latitude,#1#longitude,#1#airTemperatureAt2M,#1#windSpeedAt10M
0,49.43,-2.60,282.4,5.0
1,52.12,0.96,281.7,7.0
2,53.03,-0.50,280.7,8.0
3,53.09,-0.17,281.0,8.0
4,51.20,-1.81,281.2,5.0


### Combining header and data

Both special values may be combined in a list to extract all keys – equivalent to `"all"` but the values are explicit:

In [6]:
df_both = pdbufr.read_bufr("obs_3day.bufr", columns=["header", "data"], reader="flat")
print(f"{len(df_both)} rows, {len(df_both.columns)} columns")
# Same shape as the full extraction
assert df_both.shape == df.shape

50 rows, 103 columns


## Individual key extraction mode

When `columns` is a list of BUFR key names, only those keys are returned.  This is more efficient than block extraction when only a few fields are needed.

### Simple keys (no rank)

Keys without a rank are treated as rank 1, so `"latitude"` is equivalent to `"#1#latitude"`.

In [7]:
df_simple = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "longitude"],
    reader="flat",
)
df_simple.head()

,ident,latitude,longitude
0,03894,49.43,-2.60
1,03590,52.12,0.96
2,03379,53.03,-0.50
3,03391,53.09,-0.17
4,03743,51.20,-1.81


### Mixing header and data keys

In [8]:
df_mixed = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "typical_datetime", "latitude", "longitude", "airTemperatureAt2M", "dewpointTemperatureAt2M"],
    reader="flat",
)
df_mixed.head()

,ident,typical_datetime,latitude,longitude,airTemperatureAt2M,dewpointTemperatureAt2M
0,03894,2017-04-25 12:00:00,49.43,-2.60,282.4,274.0
1,03590,2017-04-25 12:00:00,52.12,0.96,281.7,268.4
2,03379,2017-04-25 12:00:00,53.03,-0.50,280.7,269.2
3,03391,2017-04-25 12:00:00,53.09,-0.17,281.0,270.3
4,03743,2017-04-25 12:00:00,51.20,-1.81,281.2,268.4


### Ranked keys

SYNOP messages contain cloud observations for up to eight cloud layers.  Each layer is a separate key occurrence, distinguished by the rank prefix `#N#`.  We can request specific occurrences explicitly:

In [9]:
df_cloud = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["latitude", "longitude", "cloudType", "#3#cloudType"],
    reader="flat",
    required_columns=["latitude"],
)
# cloudType (no rank) is treated as #1#cloudType
df_cloud.head()

,latitude,longitude,cloudType,#3#cloudType
0,49.43,-2.60,32,11
1,52.12,0.96,31,10
2,53.03,-0.50,38,60
3,53.09,-0.17,38,10
4,51.20,-1.81,31,10


Note that `"cloudType"` (no rank) and `"#1#cloudType"` refer to the same key – the *first* occurrence.  This differs from the ecCodes `codes_get()` function, which would return the *last* occurrence.

### Computed keys

In [10]:
df_wmo = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "WMO_station_id", "latitude", "longitude"],
    reader="flat",
)
df_wmo.head()

,ident,WMO_station_id,latitude,longitude
0,03894,3894,49.43,-2.60
1,03590,3590,52.12,0.96
2,03379,3379,53.03,-0.50
3,03391,3391,53.09,-0.17
4,03743,3743,51.20,-1.81


## Filters

### Scalar and list filters

Select a single station or a list of stations by WMO block+station number:

In [11]:
# Single station
df_st = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "longitude", "airTemperatureAt2M"],
    filters={"stationNumber": 894},
    reader="flat",
)
df_st

,ident,latitude,longitude,airTemperatureAt2M,stationNumber
0,03894,49.43,-2.6,282.4,894


In [12]:
# Multiple stations – pass a list
df_sts = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "longitude", "airTemperatureAt2M"],
    filters={"stationNumber": [894, 103]},
    reader="flat",
)
df_sts

,ident,latitude,longitude,airTemperatureAt2M,stationNumber
0,03894,49.43,-2.60,282.4,894
1,07103,48.04,-4.73,NaN,103


### Slice filter

Select the first three messages using the built-in `count` key (1-based message counter):

In [13]:
df_slice = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "longitude"],
    filters={"count": slice(1, 4)},
    reader="flat",
)
df_slice

,ident,latitude,longitude
0,03894,49.43,-2.60
1,03590,52.12,0.96
2,03379,53.03,-0.50
3,03391,53.09,-0.17


### Ranked-key filter

Filter on a specific occurrence of a repeated key using the `#N#` rank prefix.  The filter below selects messages where the **first** cloud layer is type 35 and the **second** cloud layer is type 27:

In [14]:
df_rank_f = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "longitude"],
    filters={"#1#cloudType": 35, "#2#cloudType": 27},
    reader="flat",
)
df_rank_f

,ident,latitude,longitude,#1#cloudType,#2#cloudType
0,07149,48.72,2.38,35,27
1,07145,48.77,2.01,35,27


### Tilde filter (`~`)

The leading `~` prefix matches messages where **any** occurrence of a key has the given value.  The filter below selects messages containing cloud type 62 in any layer:

In [15]:
df_tilde = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "longitude"],
    filters={"~cloudType": 62},
    reader="flat",
)
df_tilde

,ident,latitude,longitude
0,03105,55.68,-6.25
1,06252,53.22,3.22
2,06310,51.44,3.60


### Computed-key filter

Computed keys can also be used in filters.  `WMO_station_id` matches when `#1#blockNumber * 1000 + #1#stationNumber` equals the given value:

In [16]:
df_wmo_f = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "WMO_station_id", "latitude", "longitude"],
    filters={"WMO_station_id": [3894, 7103]},
    reader="flat",
)
df_wmo_f

,ident,WMO_station_id,latitude,longitude
0,03894,3894,49.43,-2.60
1,07103,7103,48.04,-4.73
